# 01 - Data Exploration

**OmniStyle Customer Intelligence Platform**

This notebook performs an initial exploration of the cleaned OmniStyle
datasets produced by the ETL pipeline (`src/data_ingestion.py` ->
`src/data_cleaning.py`). It answers the first level of business
intelligence: **What happened?**

Before running this notebook, make sure you have executed, from the
project root:

```bash
python -m src.data_ingestion
python -m src.data_cleaning
```


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_PROCESSED_DIR

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 5)


## Load processed data

In [ ]:
customers = pd.read_csv(DATA_PROCESSED_DIR / "customers_clean.csv", parse_dates=["signup_date"])
products = pd.read_csv(DATA_PROCESSED_DIR / "products_clean.csv")
orders = pd.read_csv(DATA_PROCESSED_DIR / "orders_clean.csv", parse_dates=["order_date"])
order_items = pd.read_csv(DATA_PROCESSED_DIR / "order_items_clean.csv")

print(f"customers: {customers.shape}")
print(f"products: {products.shape}")
print(f"orders: {orders.shape}")
print(f"order_items: {order_items.shape}")


## Descriptive statistics

In [ ]:
customers.describe(include="all").T


In [ ]:
orders.describe().T


## Missing values check

In [ ]:
for name, df in [("customers", customers), ("products", products),
                  ("orders", orders), ("order_items", order_items)]:
    missing = df.isna().sum()
    missing = missing[missing > 0]
    print(f"--- {name} ---")
    print(missing if len(missing) else "No missing values.")
    print()


## Revenue over time

In [ ]:
monthly_revenue = (
    orders.set_index("order_date")["total_amount"]
    .resample("MS")
    .sum()
)

ax = monthly_revenue.plot(kind="line", marker="o", title="Monthly Revenue Trend")
ax.set_ylabel("Revenue ($)")
ax.set_xlabel("Month")
plt.tight_layout()
plt.show()


## Customer distribution by region and loyalty tier

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

customers["region"].value_counts().plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Customers by Region")
axes[0].set_ylabel("Number of Customers")

tier_order = ["Bronze", "Silver", "Gold", "Platinum"]
customers["loyalty_tier"].value_counts().reindex(tier_order).plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Customers by Loyalty Tier")
axes[1].set_ylabel("Number of Customers")

plt.tight_layout()
plt.show()


## Product category performance

In [ ]:
sales = order_items.merge(products[["product_id", "category"]], on="product_id")
category_revenue = sales.groupby("category")["line_total"].sum().sort_values(ascending=False)

ax = category_revenue.plot(kind="bar", color="seagreen", title="Revenue by Category")
ax.set_ylabel("Revenue ($)")
plt.tight_layout()
plt.show()

category_revenue


## Key takeaways

- Revenue shows clear seasonality, consistent with the synthetic data
  generator's seasonal multipliers (higher volume around back-to-school and
  the November/December holiday season).
- The customer base skews toward the Bronze/Silver loyalty tiers, which is
  expected given the archetype mix used to generate the data (a majority of
  customers are new, at-risk, or inactive rather than long-tenured champions).
- Category performance varies by region due to the region/category affinity
  weighting used in data generation; see `02_customer_analysis.ipynb` for a
  deeper look at customer-level behavior.
